In [1]:
import json
import itertools
from pathlib import Path
import pandas as pd

In [2]:
def load_results_summaries(base_dir, direction_pairs, system_names):
    """
    Loads all result summaries from a directory structure.

    Args:
        base_dir (str or Path): The base directory for the evaluation outputs.
        direction_pairs (list): A list of language direction strings (e.g., 'en_de').
        system_names (list): A list of system name strings.

    Returns:
        dict: A nested dictionary containing the loaded data, structured as
              {direction: {system: [results]}}.
    """
    base_path = Path(base_dir)
    all_results = {}

    # Use itertools.product to cleanly iterate over all combinations
    for direction, system in itertools.product(direction_pairs, system_names):
        summary_path = base_path / system / direction / 'results_summary.jsonl'
        
        # Initialize the nested dictionary structure
        if direction not in all_results:
            all_results[direction] = {}

        try:
            with summary_path.open('r', encoding='utf-8') as f:
                all_results[direction][system] = [json.loads(line) for line in f]
                
        except FileNotFoundError:
            print(f"Warning: File not found, skipping: {summary_path}")
            all_results[direction][system] = None # Or [] if you prefer an empty list
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON in {summary_path}: {e}")
            all_results[direction][system] = None

    return all_results

In [3]:
def convert_results_to_dataframe(results_data):
    """
    Converts the nested dictionary of results into a single pandas DataFrame.

    Each row in the DataFrame corresponds to a single entry from a .jsonl file,
    augmented with 'direction' and 'system' columns to preserve its origin.

    Args:
        results_data (dict): The nested dictionary produced by the 
                             load_results_summaries function.

    Returns:
        pandas.DataFrame: A tidy DataFrame containing all results.
    """
    # Use a list comprehension for a fast and memory-efficient approach
    # This creates a flat list of records, where each record is a dictionary
    # that includes the original data plus the direction and system.
    all_records = [
        {
            'direction': direction,
            'system': system,
            **record  # Unpack the original record's key-value pairs
        }
        for direction, systems in results_data.items()
        for system, records in systems.items()
        if records is not None  # Gracefully skip any files that were not found
        for record in records
    ]

    if not all_records:
        print("No records were found to create a DataFrame.")
        return pd.DataFrame()

    # Convert the list of dictionaries directly into a DataFrame
    df = pd.DataFrame(all_records)

    # Reorder columns to have identifying info first, for better readability
    # Get all columns from the original data, excluding our added keys
    original_cols = [col for col in df.columns if col not in ['direction', 'system']]
    # Create the desired column order
    preferred_order = ['direction', 'system'] + original_cols
    df = df[preferred_order]

    return df

In [4]:
BASE_DIR = '/Users/javigg/Desktop/hearing2translate/evaluation/output_evals/fleurs'
DIRECTION_PAIRS = ['en_de', 'de_en', 'en_es', 'es_en', 'en_fr', 'fr_en', 'en_it', 'it_en', 'en_nl', 'en_pt', 'pt_en', 'en_zh', 'zh_en']
#SYSTEM_NAMES = ['canary-v2',  'gemma_canary-v2',  'gemma_seamlessm4t',  'owsm4.0-ctc', 'qwen2audio-7b',  'spirelm', 'tower_owsm4.0-ctc',  'tower_whisper',  'whisper',
#                'desta2-8b',  'gemma_owsm4.0-ctc',  'gemma_whisper',    'phi4multimodal',  'seamlessm4t',   'tower_canary-v2',  'tower_seamlessm4t',  'voxtral-small-24b']

#SYSTEM_NAMES = ['canary-v2',  'gemma_canary-v2',  'gemma_seamlessm4t',  'owsm4.0-ctc', 'qwen2audio-7b',  'spirelm',    'whisper',
#                'desta2-8b',  'gemma_owsm4.0-ctc',  'gemma_whisper',    'phi4multimodal',  'seamlessm4t',  'voxtral-small-24b']

SYSTEM_NAMES = ['whisper', 'seamlessm4t', 'canary-v2', 'owsm4.0-ctc',
                'aya_whisper', 'gemma_whisper', 'tower_whisper', 
                'aya_seamlessm4t', 'gemma_seamlessm4t', 'tower_seamlessm4t',
                'aya_canary-v2', 'gemma_canary-v2', 'tower_canary-v2',
                'aya_owsm4.0-ctc', 'gemma_owsm4.0-ctc', 'tower_owsm4.0-ctc',
                'desta2-8b', 'qwen2audio-7b', 'phi4multimodal', 'voxtral-small-24b', 'spirelm', 'qwen3omni'
               ]

# Call the function and store the results
results_data = load_results_summaries(BASE_DIR, DIRECTION_PAIRS, SYSTEM_NAMES)
results_df = convert_results_to_dataframe(results_data)

selected_cols = ['direction', 'system', 'SacreBLEU', 'chrF', 'LinguaPy',
                 'RefMetricX_24-Strict-linguapy', 'QEMetricX_24-Strict-linguapy',
                 'XCOMET-Strict-linguapy', 'XCOMET-QE-Strict-linguapy']
results_df = results_df[selected_cols]

In [5]:
selected_cols = ['direction', 'system', 'LinguaPy', 'QEMetricX_24-Strict-linguapy', 'XCOMET-QE-Strict-linguapy', 'chrF']
results_df = results_df[selected_cols]

In [6]:
results_df

,direction,system,LinguaPy,QEMetricX_24-Strict-linguapy,XCOMET-QE-Strict-linguapy,chrF
0,en_de,seamlessm4t,0.1560,2.4148,0.9403,61.3662
1,en_de,canary-v2,0.0000,2.3163,0.9447,61.9113
2,en_de,owsm4.0-ctc,0.1560,9.4253,0.7766,54.9522
3,en_de,aya_whisper,0.0000,1.3047,0.9666,63.3557
4,en_de,gemma_whisper,0.0000,1.3646,0.9659,62.0023
...,...,...,...,...,...,...
267,zh_en,desta2-8b,0.6349,4.3109,0.7453,46.5472
268,zh_en,qwen2audio-7b,5.0794,3.9521,0.8176,48.0031
269,zh_en,phi4multimodal,5.7143,3.7811,0.8353,51.5074
270,zh_en,voxtral-small-24b,0.2116,3.0309,0.8567,51.5150


In [7]:
lang_pairs_order = ['en_es', 'en_fr', 'en_pt', 'en_it', 'en_de', 'en_nl', 'en_zh', 'es_en', 'fr_en', 'pt_en', 'it_en', 'de_en', 'zh_en']

In [8]:
pivoted_xcomet_qe = results_df.pivot(index='system', columns='direction', values='XCOMET-QE-Strict-linguapy')[lang_pairs_order]

In [9]:
pivoted_xcomet_qe.to_csv('pivoted_xcomet_qe.csv')

In [10]:
pivoted_metricx_qe = results_df.pivot(index='system', columns='direction', values='QEMetricX_24-Strict-linguapy')[lang_pairs_order]

In [11]:
pivoted_metricx_qe.to_csv('pivoted_metricx_qe.csv')

In [12]:
pivoted_linguapy = results_df.pivot(index='system', columns='direction', values='LinguaPy')[lang_pairs_order]

In [13]:
pivoted_linguapy.to_csv('pivoted_linguapy.csv')

#### Results by language pair

In [14]:
for lang_pair in lang_pairs_order:
    lp_df = results_df.query(f"direction == '{lang_pair}'").sort_values(by = 'XCOMET-QE-Strict-linguapy').to_csv(f'fleurs_{lang_pair}.csv', index=False)

### Gender Fleurs

In [15]:
def load_all_jsons(base_dir, manifests_dir, direction_pairs, system_names):
    base_path = Path(base_dir)
    manifests_path = Path(manifests_dir)
    all_results = {}
    for direction, system in itertools.product(direction_pairs, system_names):
        results_path = base_path / system / direction / 'results.jsonl'
        direction_aux = '{direction}.jsonl'.format( direction = direction.replace('_', '-') )
        manifest_path = manifests_path / direction_aux

        # Initialize the nested dictionary structure
        if direction not in all_results:
            all_results[direction] = {}

        try:
            with results_path.open('r', encoding='utf-8') as f:
                all_results[direction][system] = [json.loads(line) for line in f]
            with manifest_path.open('r', encoding='utf-8') as f:
                manifests = [json.loads(line) for line in f]
                for it, it_manifests in zip(all_results[direction][system], manifests):
                    it_manifests['gender'] = it_manifests['benchmark_metadata']['gender']
                    it['linguapy_score'] = it['metrics']['linguapy_score'][0]
                    it['xcomet_qe_score'] = it['metrics']['xcomet_qe_score'] if it['linguapy_score'] == 0 else 0
                    it['metricx_qe_score'] = it['metrics']['metricx_qe_score'] if it['linguapy_score'] == 0 else 0
                    it.update(it_manifests)
        
        except FileNotFoundError:
            pass

        except json.JSONDecodeError as e:
            pass

    results = []
    for direction in all_results.keys():
        for system in all_results[direction].keys():
            for item in all_results[direction][system]:
                item['direction'] = direction
                item['system'] = system
                item['sample_id'] = item['benchmark_metadata']['sample_id']
                results.append(item)

    results_df = pd.DataFrame(results)

    return results_df

In [16]:
MANIFESTS_DIR = '/Users/javigg/Desktop/hearing2translate/manifests/fleurs'
BASE_DIR = '/Users/javigg/Desktop/hearing2translate/evaluation/output_evals/fleurs'

In [17]:
DIRECTION_PAIRS = ['en_de', 'de_en', 'en_es', 'es_en', 'en_fr', 'fr_en', 'en_it', 'it_en', 'en_nl', 'en_pt', 'pt_en', 'en_zh', 'zh_en']

#SYSTEM_NAMES = ['canary-v2',  'gemma_canary-v2',  'gemma_seamlessm4t',  'owsm4.0-ctc', 'qwen2audio-7b',  'spirelm', 'tower_owsm4.0-ctc',  'tower_whisper',  'whisper',
#                'desta2-8b',  'gemma_owsm4.0-ctc',  'gemma_whisper',    'phi4multimodal',  'seamlessm4t',   'tower_canary-v2',  'tower_seamlessm4t',  'voxtral-small-24b']

SYSTEM_NAMES = ['whisper', 'seamlessm4t', 'canary-v2', 'owsm4.0-ctc',
                'aya_whisper', 'gemma_whisper', 'tower_whisper', 
                'aya_seamlessm4t', 'gemma_seamlessm4t', 'tower_seamlessm4t',
                'aya_canary-v2', 'gemma_canary-v2', 'tower_canary-v2',
                'aya_owsm4.0-ctc', 'gemma_owsm4.0-ctc', 'tower_owsm4.0-ctc',
                'desta2-8b', 'qwen2audio-7b', 'phi4multimodal', 'voxtral-small-24b', 'spirelm', 'qwen3omni'
               ]

columns_to_keep = ['dataset_id', 'sample_id', 'src_lang', 'tgt_lang', 'xcomet_qe_score', 'metricx_qe_score', 'linguapy_score', 'gender', 'direction', 'system' ]
# Call the function and store the results
results_and_manifests = load_all_jsons(BASE_DIR, MANIFESTS_DIR, DIRECTION_PAIRS, SYSTEM_NAMES)[columns_to_keep]

In [18]:
results_and_manifests.head()

,dataset_id,sample_id,src_lang,tgt_lang,xcomet_qe_score,metricx_qe_score,linguapy_score,gender,direction,system
0,fleurs,1904,en,de,1.000000,0.964488,0,0,en_de,seamlessm4t
1,fleurs,1675,en,de,0.945453,3.154465,0,0,en_de,seamlessm4t
2,fleurs,1950,en,de,0.876322,5.628375,0,1,en_de,seamlessm4t
3,fleurs,1728,en,de,0.981796,1.750560,0,1,en_de,seamlessm4t
4,fleurs,1972,en,de,0.995062,0.096432,0,1,en_de,seamlessm4t


In [19]:
import io

# gender 1 --> female
# gender 0 --> male

def analyze_gender_diff_by_system(df, target_direction, target_system):
    """
    Analyzes the average difference in xcomet_qe_score between gender=0
    and gender=1 pairs for a given direction and system.
    
    Args:
        df (pd.DataFrame): The input DataFrame.
        target_direction (str): The direction to filter by (e.g., 'en_de').
        target_system (str): The system to filter by (e.g., 'canary-v2').
    """
    try:
        # Filter for the specified direction and system
        df_filtered = df[(df['direction'] == target_direction) & (df['system'] == target_system)].copy()

        if df_filtered.empty:
            print(f"No data found for direction='{target_direction}' and system='{target_system}'.")
            return

        # Separate gender 0 and gender 1
        # Ensure 'gender' is integer type for comparison
        df_filtered['gender'] = pd.to_numeric(df_filtered['gender'], errors='coerce')
        
        df_gender_0 = df_filtered[df_filtered['gender'] == 0]
        df_gender_1 = df_filtered[df_filtered['gender'] == 1]

        # Select relevant columns for merging
        cols_to_keep = ['sample_id', 'xcomet_qe_score', 'metricx_qe_score']

        # Merge to find pairs (matching sample_id)
        # This step ensures we only get sample_ids that have *both*
        # a gender=0 and a gender=1 entry.
        df_merged = pd.merge(
            df_gender_0[cols_to_keep],
            df_gender_1[cols_to_keep],
            on='sample_id',
            suffixes=('_g0', '_g1')
        )
        
        # Check if any pairs were found
        num_pairs = len(df_merged)

        if num_pairs == 0:
            print(f"No matching pairs (gender 0 & 1 for the same sample_id) found for direction='{target_direction}' and system='{target_system}'.")

        # Calculate the difference (gender 1 score - gender 0 score)
        df_merged['abs_score_diff'] = abs( df_merged['xcomet_qe_score_g1'] - df_merged['xcomet_qe_score_g0'] )
        df_merged['score_diff'] = df_merged['xcomet_qe_score_g1'] - df_merged['xcomet_qe_score_g0']  # if negative, male quality better than female quality
        
        # Calculate the average difference
        avg_diff = df_merged['score_diff'].mean()
        abs_diff = df_merged['abs_score_diff'].mean()

        # Disparity score as defined in 
        
        #Giuseppe Attanasio, Beatrice Savoldi, Dennis Fucci, and Dirk Hovy. 2024. 
        #Twists, Humps, and Pebbles: Multilingual Speech Recognition Models Exhibit 
        #Gender Performance Gaps. In Proceedings of the 2024 Conference on Empirical 
        #Methods in Natural Language Processing, pages 21318–21340, Miami, Florida, 
        #USA. Association for Computational Linguistics.
        
        phi_g0 = df_merged['xcomet_qe_score_g0'].mean()
        #print(phi_g0)
        phi_g1 = df_merged['xcomet_qe_score_g1'].mean()
        #print(phi_g1)
        E_quality = 100.0 * (phi_g0 - phi_g1) / (phi_g0 + 0.00000001)
        #print(E_quality)
        #print('----------------')

        phi_g0_metricx = 100 - 4 * df_merged['metricx_qe_score_g0'].mean()
        #print(phi_g0)
        phi_g1_metricx = 100 - 4 * df_merged['metricx_qe_score_g1'].mean()
        #print(phi_g1)
        E_quality_metricx = 100.0 * (phi_g0_metricx - phi_g1_metricx) / (phi_g0_metricx + 0.00000001)
        #print(E_quality)
        #print('----------------')

        return (avg_diff, abs_diff, E_quality, E_quality_metricx,  100*phi_g0,  100*phi_g1, 
                phi_g0_metricx, phi_g1_metricx  )

    except Exception as e:
        print(f"An error occurred during analysis for {target_direction}, {target_system}: {e}")

In [20]:
DIRECTION_PAIRS = ['en_de', 'de_en', 'en_es', 'es_en', 'en_fr', 'fr_en', 'en_it', 'it_en', 'en_nl', 'en_pt', 'pt_en', 'en_zh', 'zh_en']

#SYSTEM_NAMES = ['canary-v2',  'gemma_canary-v2',  'gemma_seamlessm4t',  'owsm4.0-ctc', 'qwen2audio-7b',  'spirelm', 'tower_owsm4.0-ctc',  'tower_whisper',  'whisper',
#                'desta2-8b',  'gemma_owsm4.0-ctc',  'gemma_whisper',    'phi4multimodal',  'seamlessm4t',   'tower_canary-v2',  'tower_seamlessm4t',  'voxtral-small-24b']

SYSTEM_NAMES = ['whisper', 'seamlessm4t', 'canary-v2', 'owsm4.0-ctc',
                'aya_whisper', 'gemma_whisper', 'tower_whisper', 
                'aya_seamlessm4t', 'gemma_seamlessm4t', 'tower_seamlessm4t',
                'aya_canary-v2', 'gemma_canary-v2', 'tower_canary-v2',
                'aya_owsm4.0-ctc', 'gemma_owsm4.0-ctc', 'tower_owsm4.0-ctc',
                'desta2-8b', 'qwen2audio-7b', 'phi4multimodal', 'voxtral-small-24b', 'spirelm', 'qwen3omni'
               ]

results_diffs = []
for direction_pair in DIRECTION_PAIRS:
    for sys in SYSTEM_NAMES:
        diff_metrics = analyze_gender_diff_by_system(results_and_manifests, direction_pair, sys)
        if diff_metrics is None:
            continue
        results_diffs.append({'system': sys, 'direction': direction_pair, 'diff_score': diff_metrics[0], 'abs_diff_score':  diff_metrics[1], 
                              'E_quality': diff_metrics[2], 'E_quality_metricx': diff_metrics[3],
                              'xcomet_qe_masculine': diff_metrics[4], 'xcomet_qe_femenine': diff_metrics[5], 
                              'metricx_qe_masculine': diff_metrics[6], 'metricx_qe_femenine': diff_metrics[7], 
                             })

df_diffs_scores = pd.DataFrame(results_diffs)

No data found for direction='en_de' and system='whisper'.
No matching pairs (gender 0 & 1 for the same sample_id) found for direction='de_en' and system='whisper'.
No matching pairs (gender 0 & 1 for the same sample_id) found for direction='de_en' and system='seamlessm4t'.
No matching pairs (gender 0 & 1 for the same sample_id) found for direction='de_en' and system='canary-v2'.
No matching pairs (gender 0 & 1 for the same sample_id) found for direction='de_en' and system='owsm4.0-ctc'.
No matching pairs (gender 0 & 1 for the same sample_id) found for direction='de_en' and system='aya_whisper'.
No matching pairs (gender 0 & 1 for the same sample_id) found for direction='de_en' and system='gemma_whisper'.
No matching pairs (gender 0 & 1 for the same sample_id) found for direction='de_en' and system='tower_whisper'.
No matching pairs (gender 0 & 1 for the same sample_id) found for direction='de_en' and system='aya_seamlessm4t'.
No matching pairs (gender 0 & 1 for the same sample_id) foun

In [21]:
lang_pairs_order = ['en_es', 'en_fr', 'en_pt', 'en_it', 'en_de', 'en_nl', 'en_zh', 'es_en', 'fr_en', 'pt_en', 'it_en', 'de_en', 'zh_en']
df_diffs_scores_pivoted = df_diffs_scores.pivot(index='system', columns='direction', values='diff_score')[lang_pairs_order]

In [22]:
df_diffs_scores_pivoted.round(4).to_csv('pivoted_diff_scores.csv')

In [23]:
df_diffs_scores_pivoted

direction,en_es,en_fr,en_pt,en_it,en_de,en_nl,en_zh,es_en,fr_en,pt_en,it_en,de_en,zh_en
system,,,,,,,,,,,,,
aya_canary-v2,0.013286,0.011394,0.024140,0.011710,0.008458,0.010485,0.008919,-0.012082,-0.004309,NaN,0.011583,NaN,-0.006729
aya_owsm4.0-ctc,0.025788,0.029694,0.014285,0.021269,0.010302,0.010580,0.020007,0.008963,-0.005228,NaN,0.006074,NaN,0.011208
aya_seamlessm4t,0.021659,0.018127,0.029858,0.016112,0.010069,-0.000610,0.015763,0.007717,-0.003629,NaN,-0.000411,NaN,0.008609
aya_whisper,0.017659,0.006221,0.007645,0.010297,0.009773,0.006217,0.012200,-0.005516,0.000144,NaN,0.022112,NaN,-0.002349
canary-v2,0.015879,0.033749,0.009841,0.011951,0.003265,0.003918,0.000000,-0.030723,0.029803,NaN,0.023716,NaN,-0.005073
desta2-8b,0.014417,0.056142,0.012440,0.031035,-0.004199,0.014174,-0.072205,-0.018370,-0.026449,NaN,0.076392,NaN,0.010992
gemma_canary-v2,0.009023,0.002264,0.007321,0.017870,0.006104,0.010356,0.004819,-0.003135,-0.005797,NaN,-0.002573,NaN,NaN
gemma_owsm4.0-ctc,-0.006907,-0.005178,0.000079,-0.010728,0.000651,-0.001167,0.006825,-0.001728,0.006568,NaN,-0.005137,NaN,-0.000041
gemma_seamlessm4t,0.020080,0.027173,0.023453,0.023105,0.014483,0.012974,0.020918,0.003178,0.013328,NaN,-0.032046,NaN,-0.003415


In [24]:
lang_pairs_order = ['en_es', 'en_fr', 'en_pt', 'en_it', 'en_de', 'en_nl', 'en_zh', 'es_en', 'fr_en', 'pt_en', 'it_en', 'de_en', 'zh_en']
df_abs_diffs_scores_pivoted = df_diffs_scores.pivot(index='system', columns='direction', values='abs_diff_score')[lang_pairs_order]

In [25]:
df_abs_diffs_scores_pivoted.round(4).to_csv('pivoted_abs_diff_scores.csv')

In [26]:
df_abs_diffs_scores_pivoted

direction,en_es,en_fr,en_pt,en_it,en_de,en_nl,en_zh,es_en,fr_en,pt_en,it_en,de_en,zh_en
system,,,,,,,,,,,,,
aya_canary-v2,0.055156,0.060650,0.064922,0.058662,0.020731,0.039742,0.053230,0.037653,0.082032,NaN,0.033767,NaN,0.071160
aya_owsm4.0-ctc,0.090868,0.093508,0.091863,0.092386,0.031145,0.057422,0.087050,0.055714,0.127136,NaN,0.072312,NaN,0.083054
aya_seamlessm4t,0.054847,0.074678,0.065108,0.063696,0.028131,0.033203,0.058997,0.046144,0.087282,NaN,0.037781,NaN,0.049637
aya_whisper,0.059054,0.073434,0.065007,0.054869,0.026768,0.037611,0.069445,0.044726,0.080604,NaN,0.044644,NaN,0.051136
canary-v2,0.069597,0.101733,0.073402,0.084761,0.030401,0.042526,0.000000,0.100852,0.111725,NaN,0.193130,NaN,0.037625
desta2-8b,0.151148,0.205906,0.114585,0.156621,0.073496,0.107222,0.280136,0.125500,0.226413,NaN,0.207310,NaN,0.174504
gemma_canary-v2,0.051917,0.076287,0.052015,0.056136,0.022506,0.039810,0.048266,0.043527,0.087751,NaN,0.054429,NaN,NaN
gemma_owsm4.0-ctc,0.026386,0.031000,0.020046,0.034995,0.011660,0.016913,0.046435,0.021671,0.024873,NaN,0.030634,NaN,0.040611
gemma_seamlessm4t,0.064996,0.088517,0.062488,0.073984,0.036085,0.043941,0.074762,0.051956,0.109751,NaN,0.068613,NaN,0.029649


In [27]:
lang_pairs_order = ['en_de', 'en_es', 'en_fr', 'en_it', 'en_nl', 'en_pt', 'en_zh', 'es_en', 'fr_en', 'it_en', 'zh_en']
# xcomet
df_xcomet_qe_masculine_scores_pivoted = df_diffs_scores.pivot(index='system', columns='direction', values='xcomet_qe_masculine')[lang_pairs_order].reindex(SYSTEM_NAMES)
df_xcomet_qe_masculine_scores_pivoted.round(4).to_csv('pivoted_xcomet_qe_masculine_scores.csv')

df_xcomet_qe_femenine_scores_pivoted = df_diffs_scores.pivot(index='system', columns='direction', values='xcomet_qe_femenine')[lang_pairs_order].reindex(SYSTEM_NAMES)
df_xcomet_qe_femenine_scores_pivoted.round(4).to_csv('pivoted_xcomet_qe_femenine_scores.csv')

### MetricX
df_metricx_qe_masculine_scores_pivoted = df_diffs_scores.pivot(index='system', columns='direction', values='metricx_qe_masculine')[lang_pairs_order].reindex(SYSTEM_NAMES)
df_metricx_qe_masculine_scores_pivoted.round(4).to_csv('pivoted_metricx_qe_masculine_scores.csv')

df_metricx_qe_femenine_scores_pivoted = df_diffs_scores.pivot(index='system', columns='direction', values='metricx_qe_femenine')[lang_pairs_order].reindex(SYSTEM_NAMES)
df_metricx_qe_femenine_scores_pivoted.round(4).to_csv('pivoted_metricx_qe_femenine_scores.csv')

In [28]:
lang_pairs_order = ['en_es', 'en_fr', 'en_pt', 'en_it', 'en_de', 'en_nl', 'en_zh', 'es_en', 'fr_en', 'pt_en', 'it_en', 'de_en', 'zh_en']
df_E_quality_scores_pivoted = df_diffs_scores.pivot(index='system', columns='direction', values='E_quality')[lang_pairs_order].reindex(SYSTEM_NAMES)

In [29]:
df_E_quality_scores_pivoted.round(4).to_csv('pivoted_E_quality_scores.csv')

In [30]:
df_E_quality_scores_pivoted

direction,en_es,en_fr,en_pt,en_it,en_de,en_nl,en_zh,es_en,fr_en,pt_en,it_en,de_en,zh_en
system,,,,,,,,,,,,,
whisper,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.122636,3.625866,NaN,-2.093257,NaN,1.321675
seamlessm4t,-2.267346,-1.433299,-1.823799,-0.280421,-0.429870,-0.066145,-3.102886,0.537600,-0.589702,NaN,0.156507,NaN,0.445625
canary-v2,-1.776991,-3.902252,-1.107797,-1.344291,-0.345558,-0.422662,0.000000,3.634491,-3.546237,NaN,-2.940353,NaN,11.229317
owsm4.0-ctc,1.559678,1.115365,52.676331,5.568599,0.224927,4.573439,-5.937800,-1.499355,14.042635,NaN,17.429963,NaN,8.600124
aya_whisper,-1.921454,-0.682012,-0.830943,-1.120184,-1.019653,-0.652602,-1.395751,0.591104,-0.015968,NaN,-2.439355,NaN,0.252609
gemma_whisper,-2.458755,-2.436957,-0.909871,-1.701380,-0.872292,-2.133136,-3.736805,0.630129,-0.650276,NaN,-1.084434,NaN,2.137496
tower_whisper,-1.705797,-2.029166,-0.600800,-1.647262,-0.670619,-1.667653,-2.300583,-0.072929,1.650861,NaN,-0.771841,NaN,2.180535
aya_seamlessm4t,-2.359691,-1.986938,-3.280215,-1.755011,-1.049414,0.063792,-1.802263,-0.841723,0.405376,NaN,0.044526,NaN,-0.927415
gemma_seamlessm4t,-2.196339,-3.001033,-2.532121,-2.535608,-1.516790,-1.363030,-2.390627,-0.347114,-1.524830,NaN,3.446840,NaN,0.375169


In [31]:
lang_pairs_order = ['en_es', 'en_fr', 'en_pt', 'en_it', 'en_de', 'en_nl', 'en_zh', 'es_en', 'fr_en', 'pt_en', 'it_en', 'de_en', 'zh_en']
df_E_quality_metricX_scores_pivoted = df_diffs_scores.pivot(index='system', columns='direction', values='E_quality_metricx')[lang_pairs_order].reindex(SYSTEM_NAMES)

In [32]:
df_E_quality_metricX_scores_pivoted.round(4).to_csv('pivoted_E_quality_metricX_scores.csv')

In [33]:
df_E_quality_metricX_scores_pivoted

direction,en_es,en_fr,en_pt,en_it,en_de,en_nl,en_zh,es_en,fr_en,pt_en,it_en,de_en,zh_en
system,,,,,,,,,,,,,
whisper,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.165498,2.137886,NaN,-0.808591,NaN,0.569264
seamlessm4t,-1.488721,-1.306090,-1.215887,-0.231225,-0.334069,-0.499159,-1.787268,-0.536562,0.122843,NaN,-0.054826,NaN,0.992656
canary-v2,-0.267610,-2.155673,-0.649592,-0.805845,0.108644,0.597538,0.000000,2.894073,-0.958271,NaN,-5.530143,NaN,-3.210786
owsm4.0-ctc,-2.967654,-3.750894,-42.689002,0.036927,-1.457479,4.647160,-3.231736,-1.897344,3.652511,NaN,3.804705,NaN,3.068974
aya_whisper,-1.186044,-0.590096,-0.667812,-0.457163,0.022995,0.147509,-1.432455,0.151259,0.666551,NaN,-1.693740,NaN,0.798848
gemma_whisper,-0.983760,-0.216125,-0.298623,-1.273482,0.051586,-0.976338,-0.624235,0.411272,-0.017279,NaN,-2.052432,NaN,0.766968
tower_whisper,-1.015274,-0.949008,-1.294261,-0.454523,-0.394814,-1.040981,-0.417601,-0.018728,0.787679,NaN,0.379576,NaN,0.912428
aya_seamlessm4t,-1.690460,-1.306923,-1.576842,-1.115479,-0.413919,0.173445,-1.037082,-0.258186,0.511851,NaN,0.252374,NaN,-0.084569
gemma_seamlessm4t,-1.433831,-1.090595,-2.703691,-1.028196,-1.372636,-0.191988,-0.700308,-0.585743,0.389439,NaN,0.670322,NaN,-0.032607


In [35]:
def df_to_latex_colored_median_anchored(blocks, colors, labels_list):
    """
    Convert DataFrames to LaTeX with Piecewise Linear Coloring.

    Logic:
    - Minimum value -> 0% intensity
    - Median value  -> 50% intensity
    - Maximum value -> 100% intensity

    This preserves magnitude within the lower and upper halves of the data
    but ensures the color scale is balanced around the median.
    """

    # --- 1. Validation ---
    if len(blocks) != len(colors):
        raise ValueError("The list of blocks and colors must have the same length.")
    if not all(len(b) == len(blocks[0]) for b in blocks):
        raise ValueError("All blocks must have the same number of rows.")

    # --- 2. Pre-calculate Stats (Min, Median, Max) ---
    # block_stats[block_idx][col_name] = (min, median, max)
    block_stats = []

    for block in blocks:
        stats = {}
        for col in block.columns:
            # Calculate the distribution points
            _min = block[col].min()
            _max = block[col].max()
            _med = block[col].median()

            stats[col] = (float(_min), float(_med), float(_max))
        block_stats.append(stats)

    # --- 3. Helper Function for Piecewise Coloring ---
    def get_cell_latex(value, minv, medv, maxv, color_name):
        value = float(value)

        # Determine Color Score based on where value sits relative to Median
        if value <= medv:
            # Lower Half: Map [min, med] -> [0, 50]
            denom = (medv - minv) if medv != minv else 1.0
            ratio = (value - minv) / denom
            score = ratio * 50
        else:
            # Upper Half: Map [med, max] -> [50, 100]
            denom = (maxv - medv) if maxv != medv else 1.0
            ratio = (value - medv) / denom
            score = 50 + (ratio * 50)

        # Safety clamp
        score = max(0, min(100, score))

        # Formatting
        if value == 0 or str(value) == 'nan':
            return f"-" # Faint dash for zero

        return f"\\cellcolor{{{color_name}!{score:.0f}}} {value:.1f}"

    all_lines = []

    # --- 4. Iterate Row-by-Row ---
    num_rows = len(blocks[0])

    for i in range(num_rows):
        row_cells = []

        # A. Label
        if i < len(labels_list):
            row_cells.append(labels_list[i])
        else:
            row_cells.append("")

        # B. Blocks
        for b_idx, block in enumerate(blocks):
            color_name = colors[b_idx]
            row_data = block.iloc[i]

            for col_name, val in row_data.items():
                # Retrieve stats
                min_v, med_v, max_v = block_stats[b_idx][col_name]

                # Generate LaTeX
                cell_latex = get_cell_latex(val, min_v, med_v, max_v, color_name)
                row_cells.append(cell_latex)

        # C. Finalize Row
        all_lines.append(" & ".join(row_cells) + r" \\")

    return "\n".join(all_lines)

In [36]:
SYSTEM_ORDER = [
    "whisper",
    "seamlessm4t",
    "canary-v2",
    "owsm4.0-ctc",

    "aya_whisper",
    "gemma_whisper",
    "tower_whisper",

    "aya_seamlessm4t",
    "gemma_seamlessm4t",
    "tower_seamlessm4t",

    "aya_canary-v2",
    "gemma_canary-v2",
    "tower_canary-v2",

    "aya_owsm4.0-ctc",
    "gemma_owsm4.0-ctc",
    "tower_owsm4.0-ctc",

    "desta2-8b",
    "qwen2audio-7b",
    "phi4multimodal",
    "voxtral-small-24b",
    "spirelm",
]

labels_list = [
    r"\cellcolor{sfmcolor} \whisper",
    r"\cellcolor{sfmcolor} \seamless",
    r"\cellcolor{sfmcolor} \canary",
    r"\cellcolor{sfmcolor} \owsm",
    r"\cellcolor{cascadecolor} \aya" + r"\ + " + r"\whisper",
    r"\cellcolor{cascadecolor} \gemma" + r"\ + " + r"\whisper",
    r"\cellcolor{cascadecolor} \tower" + r"\ + " + r"\whisper",
    r"\cellcolor{cascadecolor} \aya" + r"\ + " + r"\seamless",
    r"\cellcolor{cascadecolor} \gemma" + r"\ + " + r"\seamless",
    r"\cellcolor{cascadecolor} \tower" + r"\ + " + r"\seamless",
    r"\cellcolor{cascadecolor} \aya" + r"\ + " + r"\canary",
    r"\cellcolor{cascadecolor} \gemma" + r"\ + " + r"\canary",
    r"\cellcolor{cascadecolor} \tower" + r"\ + " + r"\canary",
    r"\cellcolor{cascadecolor} \aya" + r"\ + " + r"\owsm",
    r"\cellcolor{cascadecolor} \gemma" + r"\ + " + r"\owsm",
    r"\cellcolor{cascadecolor} \aya" + r"\ + " + r"\owsm",
    r"\cellcolor{speechllmcolor} \desta",
    r"\cellcolor{speechllmcolor} \qwenaudio",
    r"\cellcolor{speechllmcolor} \phimultimodal",
    r"\cellcolor{speechllmcolor} \voxtral",
    r"\cellcolor{speechllmcolor} \spire",
    ]

langs = [
         'en_de',
         'en_es',
         'en_fr',
         'en_it',
         'en_nl',
         'en_pt',
         'en_zh',
         'es_en',
         'fr_en',
         'it_en',
         'zh_en'
]

df_E_quality_metricX_scores_pivoted = df_E_quality_metricX_scores_pivoted.reindex(SYSTEM_ORDER).reset_index(drop=True)[langs]
df_E_quality_scores_pivoted = df_E_quality_scores_pivoted.reindex(SYSTEM_ORDER).reset_index(drop=True)[langs]
blocks = [df_E_quality_metricX_scores_pivoted, df_E_quality_scores_pivoted ]

In [37]:
colors = ['Chartreuse3', 'DarkSlateGray3']
print(df_to_latex_colored_median_anchored(blocks, colors, labels_list ))

\cellcolor{sfmcolor} \whisper & - & - & - & - & - & - & - & \cellcolor{Chartreuse3!53} 0.2 & \cellcolor{Chartreuse3!75} 2.1 & \cellcolor{Chartreuse3!45} -0.8 & \cellcolor{Chartreuse3!51} 0.6 & - & - & - & - & - & - & - & \cellcolor{DarkSlateGray3!42} 0.1 & \cellcolor{DarkSlateGray3!61} 3.6 & \cellcolor{DarkSlateGray3!43} -2.1 & \cellcolor{DarkSlateGray3!53} 1.3 \\
\cellcolor{sfmcolor} \seamless & \cellcolor{Chartreuse3!50} -0.3 & \cellcolor{Chartreuse3!39} -1.5 & \cellcolor{Chartreuse3!45} -1.3 & \cellcolor{Chartreuse3!69} -0.2 & \cellcolor{Chartreuse3!48} -0.5 & \cellcolor{Chartreuse3!51} -1.2 & \cellcolor{Chartreuse3!30} -1.8 & \cellcolor{Chartreuse3!36} -0.5 & \cellcolor{Chartreuse3!33} 0.1 & \cellcolor{Chartreuse3!53} -0.1 & \cellcolor{Chartreuse3!59} 1.0 & \cellcolor{DarkSlateGray3!55} -0.4 & \cellcolor{DarkSlateGray3!43} -2.3 & \cellcolor{DarkSlateGray3!53} -1.4 & \cellcolor{DarkSlateGray3!57} -0.3 & \cellcolor{DarkSlateGray3!59} -0.1 & \cellcolor{DarkSlateGray3!43} -1.8 & \cellc

#### Generate latex tables appendices

In [38]:
import csv
import collections
import statistics

In [39]:
data = load_results_summaries(BASE_DIR, DIRECTION_PAIRS, SYSTEM_NAMES)

In [41]:
langs = [
         'en_de',
         'en_es',
         'en_fr',
         'en_it',
         'en_nl',
         'en_pt',
         'en_zh',
         'de_en',
         'es_en',
         'fr_en',
         'it_en',
         'pt_en',
         'zh_en'
]

METRIC_TO_NAME = {
    "LinguaPy": "LinguaPy",
    "QEMetricX_24-Strict-linguapy": "MetricX$^L$",
    "XCOMET-QE-Strict-linguapy": "XCOMET$^L$",
}
metrics = METRIC_TO_NAME.keys()

SYSTEM_ORDER = [
    "whisper",
    "seamlessm4t",
    "canary-v2",
    "owsm4.0-ctc",

    "aya whisper",
    "gemma whisper",
    "tower whisper",

    "aya seamlessm4t",
    "gemma seamlessm4t",
    "tower seamlessm4t",

    "aya canary-v2",
    "gemma canary-v2",
    "tower canary-v2",

    "aya owsm4.0-ctc",
    "gemma owsm4.0-ctc",
    "tower owsm4.0-ctc",

    "desta2-8b",
    "qwen2audio-7b",
    "phi4multimodal",
    "voxtral-small-24b",
    "spirelm",
]


MAPPING_SYS_NAMES = {
    "whisper": r"\cellcolor{sfmcolor} \whisper",
    "seamlessm4t": r"\cellcolor{sfmcolor} \seamless",
    "canary-v2": r"\cellcolor{sfmcolor} \canary",
    "owsm4.0-ctc": r"\cellcolor{sfmcolor} \owsm",

    "aya whisper": r"\cellcolor{cascadecolor} \whisperfixed \,+ \aya",
    "gemma whisper": r"\cellcolor{cascadecolor} \nonefixed \,+ \gemma",
    "tower whisper": r"\cellcolor{cascadecolor} \nonefixed \,+ \tower",

    "aya seamlessm4t": r"\cellcolor{cascadecolor} \seamlessfixed \,+ \aya",
    "gemma seamlessm4t": r"\cellcolor{cascadecolor}  \nonefixed \,+ \gemma",
    "tower seamlessm4t":  r"\cellcolor{cascadecolor} \nonefixed \,+ \tower",

    "aya canary-v2":  r"\cellcolor{cascadecolor} \canaryfixed \,+ \aya",
    "gemma canary-v2": r"\cellcolor{cascadecolor} \nonefixed \,+ \gemma",
    "tower canary-v2": r"\cellcolor{cascadecolor} \nonefixed \,+ \tower",

    "aya owsm4.0-ctc": r"\cellcolor{cascadecolor} \owsmfixed \,+ \aya",
    "gemma owsm4.0-ctc": r"\cellcolor{cascadecolor} \nonefixed \,+ \gemma",
    "tower owsm4.0-ctc": r"\cellcolor{cascadecolor} \nonefixed \,+ \tower",

    "desta2-8b": r"\cellcolor{speechllmcolor} \desta",
    "qwen2audio-7b": r"\cellcolor{speechllmcolor} \qwenaudio",
    "phi4multimodal": r"\cellcolor{speechllmcolor} \phimultimodal",
    "voxtral-small-24b": r"\cellcolor{speechllmcolor} \voxtral",
    "spirelm": r"\cellcolor{speechllmcolor} \spire",
}


def printtex(*args):
    print(
        *args,
        sep=" & ",
        end=" \\\\\n",
    )

def color_cell(value, metric):
    if value  == "-":
        return "-"
    color = {
        "LinguaPy": "Brown3",
        "metricx_qe_score": "Chartreuse3",
        "QEMetricX_24-Strict-linguapy": "Chartreuse3",
        "xcomet_qe_score": "DarkSlateGray3",
        "XCOMET-QE-Strict-linguapy": "DarkSlateGray3",
    }

    s = f"{value:.1f}"
    if metric in {"LinguaPy"}:
        color = "Brown3"
        minv, maxv = 0, -20
    elif metric in {"metricx_qe_score", "QEMetricX_24-Strict-linguapy"}:
        color = "Chartreuse3"
        minv, maxv = 20, 80
    elif metric in {"xcomet_qe_score", "XCOMET-QE-Strict-linguapy"}:
        color = "DarkSlateGray3"
        minv, maxv = 20, 80
    color_v = ( (value - minv) / (maxv - minv) ) * 100
    color_v = max(0, min(100, color_v))

    return f"\\cellcolor{{{color}!{color_v:.0f}}} {s}"

print(
    r"\begin{tabular}{l" + "r" * ((len(langs)+1) * len(metrics)) + "}",
    r"\toprule",
)
printtex(
    "",
    *[
        f"\\multicolumn{{{len(langs)}}}{{c}}{{\\bf {METRIC_TO_NAME[metric]}}}"
        for metric in metrics
    ]
)
printtex(
    "",
    *[
        lang.replace('_', '-')
        for _ in metrics
        for lang in langs + [""]
    ]
)
print("\\midrule")

# invert and scale metrics
for lang in langs:
    for metric in metrics:
        for system in data[lang].keys():
            # Skip if the whole system entry is None
            if data[lang][system] is None:
                # if you want, you could also set it explicitly:
                data[lang][system] = [{"metricx_qe_score":"-", "QEMetricX_24-Strict-linguapy":"-", 
                                      "LinguaPy":"-", "xcomet_qe_score":"-", "XCOMET-QE-Strict-linguapy":"-" }]
                continue

            # Get the metric value (might be missing)
            value = data[lang][system][0].get(metric, None)

            if value is None:
                # No value for this metric -> mark as N/A
                data[lang][system][0][metric] = "N/A"
                continue

            # Now safely transform depending on metric type
            if metric in {"metricx_qe_score", "QEMetricX_24-Strict-linguapy"}:
                data[lang][system][0][metric] = 100 - 4 * value if value != "-" else "-"
            elif metric in {"LinguaPy"}:
                data[lang][system][0][metric] = -value if value != "-" else "-"
            elif metric in {"xcomet_qe_score", "XCOMET-QE-Strict-linguapy"}:
                data[lang][system][0][metric] = 100 * value if value != "-" else "-"
                
system_order = sorted(
    data[langs[0]].keys(),
    key=lambda k: SYSTEM_ORDER.index(k.replace("_", " ")),
)
for system in system_order:
    printtex(
        MAPPING_SYS_NAMES[system.replace("_", r" ")],
        *[
            color_cell(data[lang][system][0][metric], metric) if lang != "" else ""
            for metric in metrics
            for lang in langs + [""]
        ]
    )

print(r"\bottomrule \end{tabular}")

\begin{tabular}{lrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrr} \toprule
 & \multicolumn{13}{c}{\bf LinguaPy} & \multicolumn{13}{c}{\bf MetricX$^L$} & \multicolumn{13}{c}{\bf XCOMET$^L$} \\
 & en-de & en-es & en-fr & en-it & en-nl & en-pt & en-zh & de-en & es-en & fr-en & it-en & pt-en & zh-en &  & en-de & en-es & en-fr & en-it & en-nl & en-pt & en-zh & de-en & es-en & fr-en & it-en & pt-en & zh-en &  & en-de & en-es & en-fr & en-it & en-nl & en-pt & en-zh & de-en & es-en & fr-en & it-en & pt-en & zh-en &  \\
\midrule
\cellcolor{sfmcolor} \whisper & - & - & - & - & - & - & - & \cellcolor{Brown3!0} -0.0 & \cellcolor{Brown3!0} -0.0 & \cellcolor{Brown3!1} -0.1 & \cellcolor{Brown3!0} -0.0 & \cellcolor{Brown3!0} -0.0 & \cellcolor{Brown3!1} -0.2 &  & - & - & - & - & - & - & - & \cellcolor{Chartreuse3!100} 81.9 & \cellcolor{Chartreuse3!100} 85.8 & \cellcolor{Chartreuse3!100} 82.8 & \cellcolor{Chartreuse3!100} 85.4 & \cellcolor{Chartreuse3!100} 86.1 & \cellcolor{Chartreuse3!94} 76.2 &  & - & - & 

In [ ]:
color_cell(data[lang][system][0][metric], metric)

In [ ]:
data['en_es']['whisper']

In [ ]:
data[lang][system][0]

In [ ]:
import numpy as np
a = np.array([93.945,	88.752,	87.3526,	90.3912,	94.4714,	89.4673,	72.6271])/100
b = np.array([94.3489,	90.7644,	88.6046,	90.6447,	94.5339,	91.099,	74.8807])/100

In [ ]:
100*( (a - b) / a ).mean()